# Junior AI / ML Developer — müsahibəyə hazırlıq

Bu notebook vakansiyanın bəndlərinə uyğun qurulub: **Python, Pandas/NumPy, Scikit-learn, ML alqoritmləri, metrikalar, zaman sıralarının proqnozlaşdırılması, təsnifat, anomaliyaların aşkarlanması, SQL, vizuallaşdırma, Git, Deep Learning (PyTorch/TensorFlow), FastAPI/REST, modellərin inteqrasiyası**.

**Necə istifadə etməli**
1. Sualı oxu və səsli cavab verməyə çalış (müsahibədə yalnız bilmək yox, *izah etmək* vacibdir).
2. «Cavab» blokunu aç və müqayisə et.
3. Sualın altındakı kodu işə sal: nümunələr sintetik məlumatlar üzərində işləyir, internet lazım deyil.

**Lazım olan kitabxanalar:** `pip install numpy pandas scikit-learn matplotlib joblib` (istəyə görə: `seaborn torch fastapi uvicorn`).

**Mündəricat**
1. Python
2. NumPy
3. Pandas və məlumatların təmizlənməsi
4. EDA və vizuallaşdırma
5. Machine Learning nəzəriyyəsi
6. Keyfiyyət metrikaları
7. Scikit-learn praktikada (Pipeline, hiperparametrlərin seçilməsi)
8. Zaman sıralarının proqnozlaşdırılması (Time Series Forecasting)
9. Anomaliyaların aşkarlanması (Anomaly Detection)
10. Klasterləşmə
11. SQL
12. Deep Learning, PyTorch / TensorFlow, LLM
13. Modellərin inteqrasiyası: FastAPI, REST, MLOps
14. Git / GitHub
15. Keyslər və davranış sualları
16. Müsahibədən əvvəl yoxlama siyahısı

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys, time, copy, os, tempfile, sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (train_test_split, cross_val_score, StratifiedKFold,
                                     GridSearchCV, TimeSeriesSplit)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                              GradientBoostingRegressor, IsolationForest)
from sklearn.cluster import KMeans
from sklearn.dummy import DummyClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import make_classification, make_regression, make_blobs, make_moons
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score,
                             average_precision_score, mean_absolute_error, mean_squared_error,
                             r2_score, mean_absolute_percentage_error, silhouette_score)

rng = np.random.default_rng(42)
pd.set_option("display.max_columns", 20)
print("numpy", np.__version__, "| pandas", pd.__version__)

---
# 1. Python

### 1.1. `list`, `tuple`, `set`, `dict` arasında fərq nədir?

<details><summary>Cavab</summary>

| Tip | Dəyişdirilə bilən | Sıralı | Təkrarlar | `x in ...` axtarışı | Nə vaxt istifadə olunur |
|---|---|---|---|---|---|
| `list` | bəli | bəli | bəli | O(n) | elementlər ardıcıllığı |
| `tuple` | xeyr | bəli | bəli | O(n) | dəyişməz yazılar, lüğət açarları |
| `set` | bəli | xeyr | xeyr | O(1) | unikal dəyərlər, sürətli yoxlama |
| `dict` | bəli | bəli (Python 3.7-dən) | açarlar unikaldır | açara görə O(1) | açar → dəyər uyğunluğu |

`tuple` hash-lənir (elementləri hash-lənirsə), ona görə də `dict` açarı və ya `set` elementi ola bilər; `list` isə ola bilməz.
</details>

In [ ]:
items = [3, 1, 3, 2]
print("list :", items)
print("set  :", set(items))
point = (40.4, 49.9)
cache = {point: "Baku"}
print("dict :", cache)

big_list = list(range(1_000_000)); big_set = set(big_list)
t = time.perf_counter(); 999_999 in big_list; t_list = time.perf_counter() - t
t = time.perf_counter(); 999_999 in big_set;  t_set  = time.perf_counter() - t
print(f"list-də axtarış: {t_list*1e3:.3f} ms, set-də: {t_set*1e3:.4f} ms")

### 1.2. Dəyişdirilə bilən default arqumentlə bağlı problem nədir?

<details><summary>Cavab</summary>

Default dəyər funksiya təyin olunanda **bir dəfə** hesablanır. Əgər bu siyahı və ya lüğətdirsə, bütün çağırışlar eyni obyektdən istifadə edir. Düzgün yol — default olaraq `None` vermək və obyekti funksiyanın içində yaratmaqdır.
</details>

In [ ]:
def add_item_bad(item, bucket=[]):
    bucket.append(item)
    return bucket

def add_item_good(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(add_item_bad(1), add_item_bad(2))
print(add_item_good(1), add_item_good(2))

### 1.3. `==` və `is`; səthi (shallow) və dərin (deep) kopya

<details><summary>Cavab</summary>

* `==` **dəyərləri** müqayisə edir, `is` — **eyniliyi** (yaddaşda eyni obyekt olub-olmamasını). `is` `None` üçün istifadə olunur: `if x is None`.
* `copy.copy` — xarici struktur yenidir, amma daxili obyektlər ortaqdır. `copy.deepcopy` hər şeyi rekursiv kopyalayır.
</details>

In [ ]:
a, b = [1, 2], [1, 2]
print(a == b, a is b)

nested = [[1, 2], [3, 4]]
shallow, deep = copy.copy(nested), copy.deepcopy(nested)
nested[0].append(99)
print("shallow:", shallow)
print("deep   :", deep)

### 1.4. List comprehension və generator. `yield` nədir?

<details><summary>Cavab</summary>

* List comprehension `[x for x in ...]` bütün siyahını dərhal yaddaşda qurur.
* Generator ifadəsi `(x for x in ...)` və `yield` olan funksiya elementləri **tənbəl (lazy)** şəkildə, bir-bir verir — böyük fayllar və məlumat axınları ilə işləyəndə yaddaşa qənaət edir.
* Generatoru yalnız bir dəfə keçmək olar.
</details>

In [ ]:
lst = [x * x for x in range(1_000_000)]
gen = (x * x for x in range(1_000_000))
print(f"list: {sys.getsizeof(lst)/1e6:.1f} MB, generator: {sys.getsizeof(gen)} bayt")

def read_in_batches(data, batch_size):
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]

for batch in read_in_batches(list(range(10)), 4):
    print(batch)

### 1.5. `*args`, `**kwargs` və dekoratorlar

<details><summary>Cavab</summary>

* `*args` — istənilən sayda mövqe arqumentləri (tuple), `**kwargs` — adlı arqumentlər (dict).
* Dekorator — funksiyanı qəbul edib yeni, «bükülmüş» funksiya qaytaran funksiyadır. Tətbiqləri: loglama, vaxtın ölçülməsi, keşləmə (`functools.lru_cache`), veb-frameworklərdə icazələrin yoxlanması (FastAPI-də marşrutlar `@app.get` dekoratorları ilə təyin olunur).
</details>

In [ ]:
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__} {time.perf_counter() - start:.4f} saniyəyə icra olundu")
        return result
    return wrapper

@timer
def train_dummy_model(n, verbose=False):
    return sum(i * i for i in range(n))

train_dummy_model(1_000_000, verbose=True)

### 1.6. Python üzrə qısa nəzəri suallar

<details><summary>Cavablar</summary>

* **GIL (Global Interpreter Lock)** — CPython-da eyni anda yalnız bir thread bytecode icra edir. CPU-nu yükləyən tapşırıqlar üçün `multiprocessing` və ya C-də yazılmış kitabxanalar (NumPy və s. GIL-i buraxır) istifadə olunur; giriş-çıxışı gözləyən tapşırıqlar (şəbəkə, fayllar) üçün `threading` / `asyncio` uyğundur.
* **İstisnalar (exceptions):** `try / except / else / finally`; çılpaq `except:` yox, konkret istisnaları tutmaq lazımdır.
* **Kontekst meneceri (`with`)** xəta baş versə belə resursun (fayl, verilənlər bazası bağlantısı) bağlanmasına zəmanət verir.
* **Virtual mühit** (`venv`, `conda`) layihənin asılılıqlarını təcrid edir; asılılıqlar `requirements.txt` və ya `pyproject.toml`-da qeyd olunur.
* **PEP 8** — kod stili standartı; **type hints** (`def f(x: int) -> float`) oxunaqlığı artırır və FastAPI/Pydantic üçün lazımdır.
* **ML-də OOP:** sklearn sinifləri vahid `fit / predict / transform` interfeysinə əməl edir — polimorfizm nümunəsi.
</details>

---
# 2. NumPy

### 2.1. NumPy niyə təmiz Python-dan sürətlidir?

<details><summary>Cavab</summary>

* Massiv eyni tipli **fasiləsiz yaddaş blokunda** saxlanılır (obyektlərə istinadlar siyahısı kimi yox).
* Əməliyyatlar **vektorlaşdırılıb**: dövr kompilyasiya olunmuş C kodunda icra olunur, hər element üçün interpretatorun əlavə xərci yoxdur.
* Optimallaşdırılmış xətti cəbr kitabxanalarından (BLAS/LAPACK) istifadə olunur.
</details>

In [ ]:
arr = rng.random(1_000_000)
py_list = arr.tolist()

t = time.perf_counter(); s1 = sum(x * x for x in py_list); t_py = time.perf_counter() - t
t = time.perf_counter(); s2 = np.sum(arr * arr);           t_np = time.perf_counter() - t
print(f"Python: {t_py*1e3:.1f} ms | NumPy: {t_np*1e3:.1f} ms | sürətlənmə ~{t_py/t_np:.0f}x")

### 2.2. Broadcasting və `axis` nədir?

<details><summary>Cavab</summary>

**Broadcasting** — NumPy-ın müxtəlif formalı massivlər üzərində əməliyyatları kiçik massivi kopyalamadan «genişləndirərək» yerinə yetirmə qaydasıdır. Ölçülər sağdan sola müqayisə olunur və bərabər olduqda və ya biri 1 olduqda uyğun sayılır.

**`axis=0`** — sətirlər boyunca əməliyyat (nəticə hər **sütun** üçün), **`axis=1`** — sütunlar boyunca (nəticə hər **sətir** üçün).
</details>

In [ ]:
X = rng.normal(loc=10, scale=3, size=(5, 3))
mean = X.mean(axis=0)
std = X.std(axis=0)
X_scaled = (X - mean) / std
print("shape:", X.shape, mean.shape)
print("standartlaşdırmadan sonra ortalar:", X_scaled.mean(axis=0).round(6))
print("standartlaşdırmadan sonra std    :", X_scaled.std(axis=0).round(6))

### 2.3. View və copy

<details><summary>Cavab</summary>

Kəsik (`a[2:5]`) **view** qaytarır — kəsiyi dəyişmək ilkin massivi də dəyişir. Fancy indexing (`a[[1, 3]]`) və bul maskaları (`a[a > 5]`) **kopya** qaytarır. Açıq kopya — `.copy()`.
</details>

In [ ]:
a = np.arange(10)
view = a[2:5];   view[0] = 100
fancy = a[[6, 7]]; fancy[0] = -1
print(a)

---
# 3. Pandas və məlumatların təmizlənməsi

Pərakəndə satışlara aid «çirkli» dataset yaradaq: boş dəyərlər, kənar dəyərlər (outliers), təkrarlar.

In [ ]:
n = 1000
dates = pd.Timestamp("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, n), unit="D")
sales = pd.DataFrame({
    "order_id": np.arange(1, n + 1),
    "date": dates,
    "store": rng.choice(["Baku-1", "Baku-2", "Sumgait", "Ganja"], n),
    "category": rng.choice(["Electronics", "Home", "Kids", "Grocery"], n, p=[0.3, 0.3, 0.2, 0.2]),
    "price": rng.gamma(2.0, 50, n).round(2),
    "qty": rng.integers(1, 6, n),
})
sales.loc[rng.choice(n, 50, replace=False), "price"] = np.nan
sales.loc[rng.choice(n, 10, replace=False), "price"] = 99999.0
sales = pd.concat([sales, sales.sample(15, random_state=1)], ignore_index=True)
sales["revenue"] = sales["price"] * sales["qty"]
sales.head()

### 3.1. Yeni datasetlə işə nədən başlayırsan?

<details><summary>Cavab</summary>

1. `df.shape`, `df.head()`, `df.info()` — ölçü, tiplər, boş dəyərlər.
2. `df.describe()` — statistikalar, şübhəli min/max.
3. `df.isna().sum()`, `df.duplicated().sum()`.
4. Kateqoriyalar üçün `df[col].value_counts()` — yazı səhvləri, nadir dəyərlər.
5. Biznes məntiqinin yoxlanması: mənfi qiymətlər, gələcək tarixlər və s.
6. Paylanmalar və əlaqələr (histoqramlar, boxplot, korrelyasiyalar).
</details>

In [ ]:
sales.info()
print("\nBoş dəyərlər:\n", sales.isna().sum())
print("\nTəkrarlar:", sales.duplicated().sum())
sales.describe().round(1)

### 3.2. Boş dəyərlərlə (missing values) necə işləmək lazımdır?

<details><summary>Cavab</summary>

Əvvəlcə məlumatın **niyə** boş olduğunu anlamaq lazımdır (təsadüfən, yoxsa yox).
* **Silmək** — boş dəyər azdırsa və ya sütun demək olar ki, boşdursa.
* **Doldurmaq** — sabitlə, median/orta ilə (median kənar dəyərlərə daha davamlıdır), kateqoriyalar üçün moda ilə, qrup daxilində median ilə, zaman sıraları üçün interpolyasiya ilə (`ffill`, `interpolate`).
* **Modellə doldurmaq:** `KNNImputer`, `IterativeImputer`.
* `is_missing` bayraq-əlaməti əlavə etmək — boşluğun özü informativ ola bilər.
* **Vacib:** doldurma statistikaları yalnız train üzərində hesablanır (əks halda məlumat sızması baş verir), ona görə `SimpleImputer`-i `Pipeline` daxilində istifadə etmək rahatdır.
</details>

### 3.3. Kənar dəyərləri (outliers) necə tapmaq və emal etmək olar?

<details><summary>Cavab</summary>

* **IQR qaydası:** dəyər < Q1 − 1.5·IQR və ya > Q3 + 1.5·IQR olarsa, kənar dəyərdir.
* **Z-score:** |z| > 3 (normal paylanmaya yaxın məlumatlar üçün).
* Vizual: boxplot, scatter.
* Modellər: Isolation Forest, LOF (bax: bölmə 9).

Nə etməli: silmək (açıq daxiletmə xətası), məhdudlaşdırmaq (clipping/winsorizing), loqarifmləmək, davamlı modellər və metrikalar istifadə etmək (MAE, ağaclar). Kənar dəyər **dəyərli informasiya** da ola bilər (fırıldaqçılıq), ona görə qərar biznes kontekstinə əsasən verilir.
</details>

In [ ]:
clean = sales.drop_duplicates().copy()

q1, q3 = clean["price"].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
print(f"Q1={q1:.1f}, Q3={q3:.1f}, yuxarı sərhəd={upper:.1f}")
print("Açıq xətalar (99999):", (clean["price"] == 99999).sum())
print("IQR sərhədindən yuxarı:", (clean["price"] > upper).sum(), "(bir hissəsi normal bahalı mallardır!)")

clean = clean[clean["price"] != 99999].copy()
clean["price"] = clean["price"].fillna(clean.groupby("category")["price"].transform("median"))
clean["revenue"] = clean["price"] * clean["qty"]
print("Qalan sətirlər:", len(clean), "| boş dəyərlər:", clean["price"].isna().sum())

### 3.4. `loc` və `iloc`

<details><summary>Cavab</summary>

`loc` — indeksin **etiketlərinə**/sütun adlarına görə seçim (kəsik sağ sərhədi daxil edir). `iloc` — **mövqelərə** görə (Python-dakı kimi, sağ sərhəd daxil deyil).
</details>

In [ ]:
df = pd.DataFrame({"a": [10, 20, 30]}, index=[2, 0, 1])
print("loc[0]  ->", df.loc[0, "a"])
print("iloc[0] ->", df.iloc[0, 0])

### 3.5. `groupby` / `agg` — kateqoriyalar üzrə gəliri, sifariş sayını və orta çeki hesabla

In [ ]:
summary = (clean.groupby("category")
                .agg(orders=("order_id", "count"),
                     revenue=("revenue", "sum"),
                     avg_check=("revenue", "mean"))
                .sort_values("revenue", ascending=False)
                .round(1))
summary

### 3.6. `merge` (JOIN) növləri

<details><summary>Cavab</summary>

* `inner` — yalnız hər iki cədvəldə üst-üstə düşən açarlar.
* `left` — sol cədvəlin bütün sətirləri + sağdan uyğunluqlar (əks halda NaN).
* `right` — əksinə.
* `outer` — hər iki cədvəlin bütün sətirləri.

Tez-tez rast gəlinən xəta — sağ cədvəldəki təkrar açarlar sətirləri «çoxaldır». Yoxlama: `validate="many_to_one"`.
</details>

In [ ]:
stores = pd.DataFrame({"store": ["Baku-1", "Baku-2", "Sumgait", "Lankaran"],
                       "region": ["Absheron", "Absheron", "Absheron", "South"]})
for how in ["inner", "left", "right", "outer"]:
    merged = pd.merge(clean, stores, on="store", how=how, validate="many_to_one")
    print(f"{how:6s}: {merged.shape[0]} sətir, region-da boş dəyərlər: {merged['region'].isna().sum()}")

### 3.7. Pandas üzrə digər tez-tez verilən suallar

<details><summary>Cavablar</summary>

* **`apply` və vektorlaşdırma:** `apply` hər sətir üçün Python funksiyasını çağırır — yavaşdır. Vektor əməliyyatlarına üstünlük verilir (`df["a"] * df["b"]`, `np.where`, `.str`, `.dt`).
* **`SettingWithCopyWarning`:** kəsiyin kopyası ola biləcək obyektə yazma. Həll — `.loc[mask, col] = value` və ya açıq `.copy()`.
* **`pivot_table` və `groupby`:** `pivot_table` eyni aqreqasiyadır, amma nəticə «geniş» cədvəl şəklindədir.
* **`map` / `replace`** — lüğətə görə dəyərlərin əvəzlənməsi; **`astype`** / `pd.to_datetime` / `pd.to_numeric(errors="coerce")` — tiplərin çevrilməsi.
* **Böyük məlumatlar:** `read_csv`-də `chunksize`, `category` tipi, `float32`, Parquet formatı; və ya Polars / Spark / SQL.
</details>

In [ ]:
clean["is_big_order"] = np.where(clean["revenue"] > 500, "yes", "no")
clean["month"] = clean["date"].dt.to_period("M").astype(str)
pd.pivot_table(clean, index="month", columns="category", values="revenue", aggfunc="sum").round(0).head()

---
# 4. EDA və vizuallaşdırma

### 4.1. Hansı tapşırıq üçün hansı qrafik?

<details><summary>Cavab</summary>

| Tapşırıq | Qrafik |
|---|---|
| Bir rəqəmsal dəyişənin paylanması | histogram, KDE, boxplot |
| Kateqoriyaların müqayisəsi | bar chart |
| Zamana görə dinamika | line chart |
| İki rəqəmsal dəyişən arasında əlaqə | scatter plot |
| Çoxlu əlamətlərin korrelyasiyası | heatmap |
| Qruplar üzrə paylanma | boxplot / violin plot |

Alətlər: **Matplotlib** (əsas, tam nəzarət), **Seaborn** (Matplotlib üzərində statistik qrafiklər), **Plotly** (interaktiv), **Power BI** (biznes üçün dashboardlar).
Yaxşı qrafik: aydın başlıq, oxların adları və ölçü vahidləri, artıq bəzəksiz, nəticə biznes üçün ifadə olunub.
</details>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
clean["price"].plot(kind="hist", bins=40, ax=axes[0], title="Qiymətin paylanması")
axes[0].set_xlabel("qiymət, AZN")

clean.boxplot(column="price", by="category", ax=axes[1])
axes[1].set_title("Kateqoriyalar üzrə qiymət"); axes[1].set_xlabel("")

weekly = clean.set_index("date").resample("W")["revenue"].sum()
weekly.plot(ax=axes[2], title="Həftələr üzrə gəlir")
axes[2].set_ylabel("AZN")

plt.suptitle(""); plt.tight_layout(); plt.show()

In [ ]:
try:
    import seaborn as sns
    num = clean[["price", "qty", "revenue"]]
    plt.figure(figsize=(4, 3))
    sns.heatmap(num.corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1)
    plt.title("Korrelyasiyalar"); plt.show()
except ImportError:
    print("Seaborn quraşdırılmayıb — korrelyasiyalar cədvəl şəklində:")
    print(clean[["price", "qty", "revenue"]].corr().round(2))

### 4.2. Korrelyasiya = səbəbiyyət?

<details><summary>Cavab</summary>

Xeyr. Korrelyasiya **xətti** əlaqəni göstərir, amma səbəb üçüncü dəyişən (dondurma satışı və boğulma halları istidən asılıdır), təsadüf və ya tərs səbəbiyyət ola bilər. Səbəbiyyət A/B testləri və ya causal inference metodları ilə yoxlanılır. Pirson əmsalı kənar dəyərlərə həssasdır və yalnız xətti əlaqəni görür; Spirmen — monoton əlaqəni.
</details>

---
# 5. Machine Learning nəzəriyyəsi

### 5.1. Öyrənmə növləri

<details><summary>Cavab</summary>

* **Supervised (müəllimlə):** hədəf dəyişən var. **Təsnifat** (müştərinin itməsi bəli/xeyr, spam) və **reqressiya** (qiymət, satış həcmi).
* **Unsupervised (müəllimsiz):** etiketlər yoxdur. Klasterləşmə (müştəri seqmentləri), ölçünün azaldılması (PCA), anomaliyaların aşkarlanması.
* **Semi-supervised:** az etiketlənmiş + çox etiketlənməmiş məlumat.
* **Reinforcement learning:** agent mükafat vasitəsilə öyrənir (dinamik qiymətləndirmə, oyunlar).
* **Self-supervised:** etiketlər məlumatın özündən götürülür — LLM-lər belə əvvəlcədən öyrədilir (növbəti tokenin proqnozu).
</details>

### 5.2. Overfitting, underfitting, bias-variance trade-off

<details><summary>Cavab</summary>

* **Underfitting (yüksək bias):** model çox sadədir, xəta həm train-də, həm test-də yüksəkdir.
* **Overfitting (yüksək variance):** model küyü əzbərləyib; train-də xəta aşağı, test-də yüksəkdir.
* Xəta = bias² + variance + aradan qaldırıla bilməyən küy. Modelin mürəkkəbləşdirilməsi bias-ı azaldır, amma variance-ı artırır.

**Overfitting-ə qarşı:** daha çox məlumat, regulyarizasiya (L1/L2, dropout), modelin sadələşdirilməsi (`max_depth` məhdudiyyəti), əlamətlərin seçilməsi, kross-validasiya, early stopping, ansambllar (bagging).
**Underfitting-ə qarşı:** daha mürəkkəb model, yeni əlamətlər, daha az regulyarizasiya.
</details>

In [ ]:
x = np.sort(rng.uniform(0, 1, 40))
y = np.sin(2 * np.pi * x) + rng.normal(0, 0.2, 40)
x_tr, x_te, y_tr_, y_te_ = train_test_split(x.reshape(-1, 1), y, test_size=0.3, random_state=0)

for degree in [1, 4, 15]:
    m = make_pipeline(PolynomialFeatures(degree), LinearRegression()).fit(x_tr, y_tr_)
    tr = mean_squared_error(y_tr_, m.predict(x_tr))
    te = mean_squared_error(y_te_, m.predict(x_te))
    verdict = "underfit" if degree == 1 else ("good fit" if degree == 4 else "overfit")
    print(f"degree={degree:2d}: train MSE={tr:.3f}, test MSE={te:.3f}  -> {verdict}")

### 5.3. Məlumatları niyə train / validation / test-ə bölürük? Data leakage nədir?

<details><summary>Cavab</summary>

* **Train** — öyrətmə, **validation** — hiperparametrlərin seçilməsi və modelin seçimi, **test** — yekun ədalətli qiymətləndirmə (bir dəfə istifadə olunur).
* **Data leakage (məlumat sızması)** — öyrətməyə proqnoz anında mövcud olmayan informasiya düşür. Nümunələr:
  * bölmədən əvvəl bütün datasetdə miqyaslama/boşluqların doldurulması;
  * hadisədən sonra yaranan əlamət (məsələn, malın qaytarılmasını proqnozlaşdırarkən «qaytarılma tarixi»);
  * zaman sırasının təsadüfi qarışdırılması — model «gələcəyi görür»;
  * eyni müştərilər həm train-də, həm test-də (`GroupKFold` lazımdır).
* Sızmanın əlaməti — şübhəli dərəcədə yüksək keyfiyyət (məsələn, ROC-AUC 0.99).
</details>

### 5.4. Kross-validasiya

<details><summary>Cavab</summary>

**K-Fold:** məlumat K hissəyə bölünür, model K dəfə öyrədilir, hər dəfə bir hissə validasiya olur. Nəticə — metrikanın ortası ± standart sapması. Bu, bir bölmədən daha etibarlı qiymətləndirmə verir.
* `StratifiedKFold` — siniflərin payını saxlayır (balanssız təsnifat üçün).
* `GroupKFold` — eyni qrupun (müştərinin) obyektləri eyni zamanda train-də və val-da olmur.
* `TimeSeriesSplit` — zaman sıraları üçün: keçmişdə öyrənib gələcəkdə yoxlayırıq.
</details>

In [ ]:
tscv = TimeSeriesSplit(n_splits=4)
for i, (tr_idx, val_idx) in enumerate(tscv.split(np.arange(20))):
    print(f"fold {i}: train={tr_idx.min()}..{tr_idx.max()}  val={val_idx.min()}..{val_idx.max()}")

### 5.5. Regulyarizasiya: L1 (Lasso) və L2 (Ridge)

<details><summary>Cavab</summary>

Regulyarizasiya itki funksiyasına çəkilərin böyüklüyünə görə cərimə əlavə edir və overfitting-i azaldır.
* **L1 (Lasso):** cərimə `α·Σ|w|`. Əmsalların bir hissəsini sıfırlayır → daxili **əlamət seçimi**.
* **L2 (Ridge):** cərimə `α·Σw²`. Çəkiləri kiçildir, amma sıfırlamır; multikollinearlıqla yaxşı işləyir.
* **Elastic Net** — L1 və L2-nin kombinasiyası.
* `α` nə qədər böyükdürsə (sklearn-də LogisticRegression üçün — `C` nə qədər kiçikdirsə), regulyarizasiya bir o qədər güclüdür.
* Regulyarizasiyadan əvvəl əlamətləri **miqyaslamaq** lazımdır.
</details>

In [ ]:
Xr, yr = make_regression(n_samples=200, n_features=20, n_informative=5, noise=10, random_state=42)
for model in [LinearRegression(), Ridge(alpha=10), Lasso(alpha=1.0)]:
    model.fit(Xr, yr)
    zeros = np.sum(np.abs(model.coef_) < 1e-6)
    print(f"{type(model).__name__:16s} sıfır əmsallar: {zeros:2d} / 20")

### 5.6. Əlamətlərin miqyaslanması və kodlaşdırılması

<details><summary>Cavab</summary>

**Miqyaslama** (`StandardScaler` — orta 0, std 1; `MinMaxScaler` — [0, 1] aralığına; `RobustScaler` — kənar dəyərlərə davamlı).
* **Lazımdır:** regulyarizasiyalı xətti modellər, KNN, SVM, K-means, PCA, neyron şəbəkələri — məsafəyə və ya qradiyent enişinə əsaslanan hər şey.
* **Lazım deyil:** qərar ağacları, Random Forest, gradient boosting.

**Kateqorial əlamətlər:**
* **One-Hot Encoding** — az sayda dəyəri olan nominal kateqoriyalar üçün.
* **Ordinal / Label Encoding** — sıralı kateqoriyalar üçün (S < M < L); ağaclar üçün nominallarda da yol veriləndir.
* **Target encoding** — kateqoriya üzrə hədəfin orta dəyəri; çox sayda dəyəri olan əlamətlər üçün; ciddi şəkildə yalnız train-də/fold-lar daxilində hesablanır, əks halda sızma olur.
* CatBoost və LightGBM kateqoriyalarla özləri işləyə bilir.
</details>

### 5.7. Əsas alqoritmlər — qısaca

<details><summary>Cavab</summary>

| Alqoritm | İdeya | Üstünlüklər | Çatışmazlıqlar |
|---|---|---|---|
| Linear Regression | əlamətlərin xətti kombinasiyası, MSE-nin minimallaşdırılması | sürətli, interpretasiya olunan | yalnız xətti asılılıqlar |
| Logistic Regression | xətti model + siqmoid → sinif ehtimalı | sürətli, ehtimallar, interpretasiya | qeyri-xəttiliyi əlamətlərlə əlavə etmək lazımdır |
| Decision Tree | əlamətlər üzrə ardıcıl bölmələr (Gini / entropiya / MSE) | interpretasiya olunan, miqyaslama lazım deyil | asanlıqla overfit olur |
| Random Forest | bootstrap nümunələri və əlamətlərin təsadüfi alt çoxluqları üzərində çoxlu ağac, ortalama (**bagging**) | davamlı, az tənzimləmə | ağırdır, ekstrapolyasiyada zəifdir |
| Gradient Boosting (XGBoost, LightGBM, CatBoost) | ağaclar ardıcıl qurulur, hər biri əvvəlkilərin xətalarını düzəldir (**boosting**) | adətən cədvəl məlumatlarında ən yaxşısı | tənzimləmə lazımdır, overfit ola bilər |
| KNN | sinif K ən yaxın qonşuya görə | sadə | böyük məlumatlarda yavaş, miqyaslama lazımdır |
| SVM | maksimal boşluqlu ayırıcı hipermüstəvi, qeyri-xəttilik üçün nüvələr | orta ölçülü məlumatlarda yaxşıdır | pis miqyaslanır |
| Naive Bayes | Bayes teoremi + əlamətlərin müstəqilliyi | çox sürətli, mətnlər | sadəlövh fərziyyə |
| K-Means | sentroidlər ətrafında klasterlər | sadə | K-nı vermək lazımdır, sferik klasterlər |
</details>

### 5.8. Bagging və Boosting

<details><summary>Cavab</summary>

* **Bagging** (Random Forest): modellər bootstrap nümunələrində **paralel və müstəqil** öyrədilir, nəticələr ortalanır → **variance** azalır.
* **Boosting** (AdaBoost, Gradient Boosting): modellər **ardıcıl** öyrədilir, hər növbəti əvvəlkilərin xətalarını düzəldir → **bias** azalır.
* **Stacking:** meta-model bir neçə baza modelin proqnozları üzərində öyrədilir.
</details>

### 5.9. Qradiyent enişi (Gradient Descent)

<details><summary>Cavab</summary>

İterativ optimallaşdırma: çəkilər itki funksiyasının antiqradiyenti istiqamətində hərəkət edir: `w = w − lr · ∂L/∂w`.
* **Learning rate** çox böyükdürsə → dağılır; çox kiçikdirsə → yavaş yığılır.
* **Batch GD** (bütün nümunə), **SGD** (bir obyekt), **Mini-batch** (batch-lər üzrə — DL-də standart).
* Təkmilləşdirilmiş optimallaşdırıcılar: Momentum, RMSProp, **Adam**.
</details>

### 5.10. Ölçü lənəti (curse of dimensionality) və PCA

<details><summary>Cavab</summary>

Əlamətlərin sayı artdıqca məlumatlar seyrəkləşir, nöqtələr arasındakı məsafələr «bərabərləşir» — KNN və klasterləşmə əziyyət çəkir, overfitting riski artır.
**PCA** maksimal dispersiyanın ortoqonal istiqamətlərini tapır və məlumatları ilk k komponentə proyeksiya edir. Sıxılma, vizuallaşdırma və multikollinearlıqla mübarizə üçün tətbiq olunur. PCA-dan əvvəl məlumatlar standartlaşdırılır. Digər metodlar: t-SNE, UMAP (vizuallaşdırma üçün), əlamət seçimi.
</details>

---
# 6. Keyfiyyət metrikaları

### 6.1. Təsnifat metrikaları. Accuracy niyə aldada bilər?

<details><summary>Cavab</summary>

Xəta matrisi (confusion matrix): **TP, FP, FN, TN**.
* **Accuracy** = (TP+TN) / hamısı. Balanssız məlumatlarda faydasızdır: sıfırıncı sinif 95% olduqda «həmişə 0» deyən model 95% verir.
* **Precision** = TP / (TP+FP) — müsbət proqnozlaşdırılanlardan neçəsi düzdür. **Yalançı həyəcan** baha başa gəldikdə vacibdir (dürüst əməliyyatın bloklanması).
* **Recall** = TP / (TP+FN) — müsbətlərin hansı hissəsini tapdıq. **Buraxma** baha başa gəldikdə vacibdir (fırıldaqçılıq, xəstəlik).
* **F1** — precision və recall-un harmonik ortası.
* **ROC-AUC** — təsadüfi müsbət obyektin təsadüfi mənfidən yüksək score alması ehtimalı; həddən asılı deyil.
* **PR-AUC (average precision)** — güclü balanssızlıqda ROC-AUC-dən daha informativdir.
</details>

In [ ]:
Xc, yc = make_classification(n_samples=3000, n_features=10, n_informative=5,
                             weights=[0.95], random_state=42)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=42)
print("1-ci sinfin payı:", yc.mean().round(3))

dummy = DummyClassifier(strategy="most_frequent").fit(Xc_tr, yc_tr)
print("Dummy accuracy:", accuracy_score(yc_te, dummy.predict(Xc_te)).round(3), "— amma recall = 0!")

clf = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xc_tr, yc_tr)
pred = clf.predict(Xc_te)
proba = clf.predict_proba(Xc_te)[:, 1]
print("\nConfusion matrix [[TN FP] [FN TP]]:\n", confusion_matrix(yc_te, pred))
print(classification_report(yc_te, pred, digits=3))
print(f"ROC-AUC = {roc_auc_score(yc_te, proba):.3f} | PR-AUC = {average_precision_score(yc_te, proba):.3f}")

### 6.2. Təsnifat həddini (threshold) necə seçmək olar?

<details><summary>Cavab</summary>

Default olaraq 0.5-dir, amma hədd **biznes qərarıdır**. Həddi artırmaq → precision yüksəlir, recall azalır. Hədd validation üzərində xətaların dəyərinə görə (məsələn, FN 100 AZN, FP 5 AZN) və ya hədəf recall-a görə seçilir.
</details>

In [ ]:
for t in [0.3, 0.5, 0.7, 0.9]:
    p = (proba >= t).astype(int)
    print(f"threshold={t}: precision={precision_score(yc_te, p, zero_division=0):.3f}, "
          f"recall={recall_score(yc_te, p):.3f}, f1={f1_score(yc_te, p):.3f}")

### 6.3. Balanssız siniflərlə necə işləmək lazımdır?

<details><summary>Cavab</summary>

1. Düzgün metrikalar: accuracy yox, precision/recall/F1, PR-AUC.
2. Bölmədə `stratify`, `StratifiedKFold`.
3. Sinif çəkiləri: `class_weight="balanced"`, XGBoost-da `scale_pos_weight`.
4. **Yalnız train-də** resampling: undersampling, oversampling, SMOTE (`imbalanced-learn`).
5. Həddin seçilməsi.
6. Müsbət nümunələr çox azdırsa — anomaliya axtarışı kimi qoyuluş.
</details>

### 6.4. Reqressiya metrikaları

<details><summary>Cavab</summary>

* **MAE** — orta mütləq xəta, hədəfin vahidlərində, kənar dəyərlərə davamlıdır.
* **MSE / RMSE** — böyük xətaları daha çox cərimələyir; RMSE hədəfin vahidlərindədir.
* **R²** — izah olunan dispersiyanın payı (1 — ideal, 0 — orta ilə proqnoz kimi, < 0 ola bilər).
* **MAPE** — orta faiz xətası, biznes üçün başa düşüləndir, amma hədəf 0-a yaxın olanda «partlayır». Alternativlər: **WAPE**, sMAPE.
</details>

In [ ]:
y_true = np.array([100, 150, 200, 250, 1000])
y_pred = np.array([110, 140, 210, 240, 600])
print(f"MAE  = {mean_absolute_error(y_true, y_pred):.1f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_true, y_pred)):.1f}   <- böyük xəta üstünlük təşkil edir")
print(f"R2   = {r2_score(y_true, y_pred):.3f}")
print(f"MAPE = {mean_absolute_percentage_error(y_true, y_pred):.1%}")
print(f"WAPE = {np.abs(y_true - y_pred).sum() / y_true.sum():.1%}")

---
# 7. Scikit-learn praktikada

### 7.1. `fit`, `transform`, `fit_transform`, `predict` — fərq nədir?

<details><summary>Cavab</summary>

* `fit` — parametrləri öyrənmək (scaler üçün orta/std, model üçün çəkilər) **yalnız train-də**.
* `transform` — öyrənilmiş çevirməni tətbiq etmək (train, val, test-də).
* `fit_transform` — hər iki addım birlikdə, **yalnız train-də**. Test-də `fit_transform` çağırmaq klassik xətadır (sızma və uyğunsuzluq).
* `predict` / `predict_proba` — modelin proqnozu.
</details>

### 7.2. `Pipeline` və `ColumnTransformer` nəyə lazımdır?

<details><summary>Cavab</summary>

* İlkin emalı və modeli **bir obyektdə** birləşdirirlər: kross-validasiyada sızma olmur (scaler hər fold daxilində öyrədilir), bütün addımların hiperparametrlərini seçmək rahatdır, saxlama və deploy üçün bir obyekt.
* `ColumnTransformer` müxtəlif sütunlara müxtəlif çevirmələr tətbiq edir (rəqəmsal — imputer + scaler, kateqorial — one-hot).
</details>

Nümunə: pərakəndə şəbəkədə **müştəri itkisi** (churn) modeli.

In [ ]:
n = 3000
churn = pd.DataFrame({
    "tenure_months": rng.integers(1, 60, n),
    "monthly_spend": rng.normal(80, 30, n).clip(5),
    "purchases_last_90d": rng.poisson(4, n),
    "city": rng.choice(["Baku", "Ganja", "Sumgait"], n),
    "has_loyalty_card": rng.choice(["yes", "no"], n),
})
logit = (1.5 - 0.05 * churn["tenure_months"] - 0.3 * churn["purchases_last_90d"]
         + 0.8 * (churn["has_loyalty_card"] == "no"))
churn["churn"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
churn.loc[rng.choice(n, 100, replace=False), "monthly_spend"] = np.nan
print("İtki payı:", churn["churn"].mean().round(3))
churn.head()

In [ ]:
num_cols = ["tenure_months", "monthly_spend", "purchases_last_90d"]
cat_cols = ["city", "has_loyalty_card"]
X = churn[num_cols + cat_cols]
y = churn["churn"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

preprocess = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scaler", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "GradBoosting": GradientBoostingClassifier(random_state=42),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    scores = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring="roc_auc")
    print(f"{name:13s} ROC-AUC = {scores.mean():.3f} ± {scores.std():.3f}")

### 7.3. Hiperparametrləri necə seçmək olar?

<details><summary>Cavab</summary>

* **Parametrlər** öyrətmə zamanı öyrənilir (çəkilər), **hiperparametrlər** əvvəlcədən verilir (`max_depth`, `C`, `learning_rate`).
* `GridSearchCV` — bütün kombinasiyaların yoxlanması; `RandomizedSearchCV` — təsadüfi kombinasiyalar (böyük fəzalarda daha sürətli); **Optuna** — Bayes optimallaşdırması.
* Seçim yalnız train üzərində kross-validasiya ilə aparılır, test toxunulmaz qalır.
</details>

In [ ]:
pipe = Pipeline([("prep", preprocess), ("model", RandomForestClassifier(random_state=42))])
param_grid = {"model__n_estimators": [100, 300], "model__max_depth": [3, 6, None]}
grid = GridSearchCV(pipe, param_grid, cv=3, scoring="roc_auc", n_jobs=-1)
grid.fit(X_tr, y_tr)
print("Ən yaxşı parametrlər:", grid.best_params_, "| CV ROC-AUC:", round(grid.best_score_, 3))
print("Test ROC-AUC        :", round(roc_auc_score(y_te, grid.predict_proba(X_te)[:, 1]), 3))

### 7.4. Modeli / əlamətlərin əhəmiyyətini necə interpretasiya etmək olar?

<details><summary>Cavab</summary>

* Xətti modelin əmsalları (miqyaslanmış əlamətlərdə).
* Ağaclarda `feature_importances_` (impurity-based; çox unikal dəyəri olan əlamətlərin əhəmiyyətini şişirdir).
* `permutation_importance` — əlamət qarışdırılanda metrikanın düşməsi; daha ədalətlidir.
* **SHAP** — hər əlamətin hər ayrıca proqnoza töhfəsi; biznesə izah etmək üçün yaxşıdır.
</details>

In [ ]:
best = grid.best_estimator_
feature_names = best.named_steps["prep"].get_feature_names_out()
importances = pd.Series(best.named_steps["model"].feature_importances_, index=feature_names).sort_values()
importances.plot(kind="barh", figsize=(6, 3.5), title="Feature importance (Random Forest)")
plt.tight_layout(); plt.show()

### 7.5. Modeli tətbiqdə istifadə üçün necə saxlamaq olar?

<details><summary>Cavab</summary>

sklearn üçün `joblib.dump` / `pickle` (yalnız modeli yox, **bütün Pipeline-ı** saxlayırıq). Dillər və platformalar arasında daşınma üçün — **ONNX**. PyTorch-da — `torch.save(model.state_dict())`. Kitabxanaların versiyalarını mütləq qeyd etmək lazımdır: sklearn-in bir versiyasında saxlanılmış model digərində yüklənməyə bilər. Eksperimentlərin və model versiyalarının uçotu üçün — **MLflow**.
</details>

In [ ]:
import joblib
model_path = os.path.join(tempfile.gettempdir(), "churn_model.joblib")
joblib.dump(best, model_path)
loaded = joblib.load(model_path)
new_customers = pd.DataFrame([
    {"tenure_months": 2, "monthly_spend": 40, "purchases_last_90d": 0, "city": "Baku", "has_loyalty_card": "no"},
    {"tenure_months": 48, "monthly_spend": 120, "purchases_last_90d": 9, "city": "Ganja", "has_loyalty_card": "yes"},
])
print("İtki ehtimalı:", loaded.predict_proba(new_customers)[:, 1].round(3))

---
# 8. Zaman sıralarının proqnozlaşdırılması (Time Series Forecasting)

### 8.1. Zaman sıraları adi cədvəl məlumatlarından nə ilə fərqlənir?

<details><summary>Cavab</summary>

* Müşahidələr **zamandan və bir-birindən asılıdır** (avtokorrelyasiya) — qarışdırmaq olmaz, bölmə yalnız zamana görə.
* Komponentlər: **trend**, **mövsümilik** (həftəlik, illik), bayramlar/aksiyalar, küy.
* **Stasionarlıq** — orta və dispersiya zamanla sabitdir; klassik modellərə (ARIMA) lazımdır. Yoxlama — Dikki-Fuller testi (ADF); gətirmə — fərqləndirmə (differencing), loqarifmləmə.
* Əlamətlər yalnız keçmişdən qurulur: lag-lar, sürüşən ortalar, təqvim.
</details>

### 8.2. Hansı proqnozlaşdırma yanaşmalarını bilirsən?

<details><summary>Cavab</summary>

1. **Baseline:** naive (sabah = bu gün), seasonal naive (= keçən həftənin eyni günü), sürüşən orta. Model bunları mütləq üstələməlidir.
2. **Statistik:** eksponensial hamarlama (Holt-Winters), **ARIMA / SARIMA** (`statsmodels`), **Prophet**.
3. **ML:** lag və təqvim əlamətləri üzərində gradient boosting (LightGBM) — çoxlu mal və mağazası olan pərakəndə üçün çox vaxt ən yaxşı variantdır.
4. **Deep Learning:** LSTM/GRU, TCN, Temporal Fusion Transformer, N-BEATS; zaman sıraları üçün foundation modellər.

Metrikalar: MAE, RMSE, MAPE/WAPE. Validasiya — `TimeSeriesSplit` / sürüşən pəncərə ilə backtesting.
</details>

Nümunə: trend, həftəlik və illik mövsümiliyi olan gündəlik satışların proqnozu.

In [ ]:
idx = pd.date_range("2023-01-01", "2025-06-30", freq="D")
t = np.arange(len(idx))
y_ts = (200 + 0.1 * t
        + 30 * np.sin(2 * np.pi * t / 7)
        + 40 * np.sin(2 * np.pi * t / 365.25)
        + rng.normal(0, 10, len(idx)))
ts = pd.DataFrame({"sales": y_ts}, index=idx)

def make_features(df):
    out = df.copy()
    out["dayofweek"] = out.index.dayofweek
    out["month"] = out.index.month
    out["dayofyear"] = out.index.dayofyear
    for lag in [7, 14, 28, 364]:
        out[f"lag_{lag}"] = out["sales"].shift(lag)
    out["rolling_mean_7"] = out["sales"].shift(7).rolling(7).mean()
    return out.dropna()

feat = make_features(ts)
split_date = "2025-04-01"
train, test = feat[feat.index < split_date], feat[feat.index >= split_date]
feature_cols = [c for c in feat.columns if c != "sales"]

gbr = GradientBoostingRegressor(random_state=42).fit(train[feature_cols], train["sales"])
test_pred = gbr.predict(test[feature_cols])
seasonal_naive = test["lag_7"]

print(f"Seasonal naive MAE: {mean_absolute_error(test['sales'], seasonal_naive):.2f}")
print(f"GradBoosting  MAE: {mean_absolute_error(test['sales'], test_pred):.2f}")

plt.figure(figsize=(14, 4))
plt.plot(train.index[-120:], train["sales"].iloc[-120:], label="train")
plt.plot(test.index, test["sales"], label="fakt")
plt.plot(test.index, test_pred, label="GBR proqnozu")
plt.title("Satış proqnozu"); plt.legend(); plt.show()

**Müsahibə üçün şərh:** bütün əlamətlər ən azı 7 gün əvvəlki məlumatdan istifadə edir, ona görə də qiymətləndirmə bir həftə qabağa proqnoz üçün ədalətlidir. 7 gündən uzun horizont üçün ya daha uzun lag-lar (direct strategiya), ya da rekursiv proqnoz lazımdır. Real pərakəndə məlumatlarında bayramlar (Novruz, Yeni il), aksiyalar, qiymətlər, hava, mal qalıqları əlavə olunur (out-of-stock tələbi təhrif edir).

---
# 9. Anomaliyaların aşkarlanması (Anomaly Detection)

### 9.1. Anomaliya aşkarlamanın hansı metodlarını bilirsən?

<details><summary>Cavab</summary>

* **Statistik:** z-score, IQR, sürüşən orta ± k·std qaydaları (sıralar üçün).
* **Müəllimsiz ML:**
  * **Isolation Forest** — anomaliyalar təsadüfi bölmələrlə daha tez təcrid olunur (ağacda yol daha qısadır);
  * **LOF (Local Outlier Factor)** — nöqtənin sıxlığı qonşularınkından aşağıdır;
  * **One-Class SVM**;
  * **DBSCAN** — klasterlərdən kənar nöqtələr.
* **Deep Learning:** autoencoder-lər (yüksək rekonstruksiya xətası = anomaliya).
* **Müəllimlə:** etiketlənmiş fırıldaqçılıq halları varsa — balanssız təsnifat.

**Etiketsiz necə qiymətləndirmək olar:** ən şübhəli top-N halların ekspertlər tərəfindən əl ilə yoxlanması, precision@k, nəticələrin sabitliyi, biznesdən əks-əlaqə. `contamination` parametri — anomaliyaların gözlənilən payı.

**Pərakəndədə tətbiqlər:** fırıldaqçı əməliyyatlar, qiymətlərdə xətalar, anomal qaytarmalar, satışlarda kəskin artım və ya düşüş, avadanlıq nasazlıqları.
</details>

In [ ]:
normal_tx = np.column_stack([rng.normal(60, 20, 980).clip(1), rng.poisson(3, 980) + 1])
anomal_tx = np.column_stack([rng.uniform(400, 800, 20), rng.integers(15, 30, 20)])
tx = pd.DataFrame(np.vstack([normal_tx, anomal_tx]), columns=["amount", "items"])
tx["is_fraud"] = [0] * 980 + [1] * 20

iso = IsolationForest(contamination=0.02, random_state=42)
tx["iso_flag"] = (iso.fit_predict(tx[["amount", "items"]]) == -1).astype(int)

z = (tx["amount"] - tx["amount"].mean()) / tx["amount"].std()
tx["z_flag"] = (z.abs() > 3).astype(int)

for col in ["iso_flag", "z_flag"]:
    print(f"{col:9s} precision={precision_score(tx['is_fraud'], tx[col]):.2f}, "
          f"recall={recall_score(tx['is_fraud'], tx[col]):.2f}")

plt.figure(figsize=(6, 4))
plt.scatter(tx["amount"], tx["items"], c=tx["iso_flag"], cmap="coolwarm", s=12)
plt.xlabel("çekin məbləği, AZN"); plt.ylabel("çekdə mal sayı"); plt.title("Isolation Forest: qırmızılar — anomaliyalar")
plt.show()

---
# 10. Klasterləşmə

### 10.1. K-Means necə işləyir və klaster sayını necə seçmək olar?

<details><summary>Cavab</summary>

1. K təsadüfi sentroid seçirik (sklearn-də — k-means++).
2. Hər nöqtəni ən yaxın sentroida aid edirik.
3. Sentroidləri klaster nöqtələrinin ortası kimi yenidən hesablayırıq.
4. Yığılana qədər təkrarlayırıq.

K-nın seçimi: **dirsək metodu** (inertia), **silhouette score** (−1-dən 1-ə qədər, nə qədər böyükdürsə o qədər yaxşı), biznes mənası (marketinq real olaraq neçə seqmentdən istifadə edə bilər). Məlumatlar miqyaslanmalıdır. Alternativlər: DBSCAN (istənilən forma, kənar dəyərləri özü tapır), iyerarxik klasterləşmə, Gaussian Mixture.

**Nümunə:** müştərilərin RFM seqmentasiyası (Recency, Frequency, Monetary).
</details>

In [ ]:
rfm, _ = make_blobs(n_samples=600, centers=[[10, 20, 500], [60, 5, 150], [200, 1, 50]],
                    cluster_std=[5, 3, 60], random_state=42)
rfm = pd.DataFrame(rfm, columns=["recency_days", "frequency", "monetary"])
rfm_scaled = StandardScaler().fit_transform(rfm)

for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(rfm_scaled)
    print(f"k={k}: inertia={km.inertia_:7.1f}, silhouette={silhouette_score(rfm_scaled, km.labels_):.3f}")

rfm["segment"] = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(rfm_scaled)
rfm.groupby("segment").mean().round(1)

---
# 11. SQL

Məlumatlarımızı yaddaşdakı SQLite-a yükləyib məşq edək.

In [ ]:
con = sqlite3.connect(":memory:")
clean.assign(date=clean["date"].dt.strftime("%Y-%m-%d")).to_sql("sales", con, index=False)
sales.assign(date=sales["date"].dt.strftime("%Y-%m-%d")).to_sql("sales_raw", con, index=False)
stores.to_sql("stores", con, index=False)

def q(sql):
    return pd.read_sql_query(sql, con)

q("SELECT * FROM sales LIMIT 3")

### 11.1. SQL sorğusunun icra ardıcıllığı. `WHERE` və `HAVING`

<details><summary>Cavab</summary>

Məntiqi ardıcıllıq: **FROM → JOIN → WHERE → GROUP BY → HAVING → SELECT → DISTINCT → ORDER BY → LIMIT**.
* `WHERE` qruplaşdırmadan **əvvəl sətirləri** süzür (orada aqreqatlardan istifadə etmək olmaz).
* `HAVING` aqreqasiyadan **sonra qrupları** süzür.
* Buna görə `SELECT`-dəki aliası `WHERE`-də istifadə etmək olmaz (əksər VBİS-lərdə), amma `ORDER BY`-da olar.
</details>

In [ ]:
q("""
SELECT category,
       COUNT(*)               AS orders,
       ROUND(SUM(revenue), 0) AS revenue
FROM sales
WHERE qty >= 2                       -- sətirlərin süzülməsi
GROUP BY category
HAVING SUM(revenue) > 20000          -- qrupların süzülməsi
ORDER BY revenue DESC
""")

### 11.2. JOIN növləri

<details><summary>Cavab</summary>

`INNER JOIN` — yalnız uyğunluqlar; `LEFT JOIN` — sol cədvəlin bütün sətirləri; `RIGHT JOIN`; `FULL OUTER JOIN` — hər ikisinin bütün sətirləri; `CROSS JOIN` — dekart hasili; `SELF JOIN` — cədvəlin özü ilə birləşməsi (işçi → rəhbər).
Tapşırıq: «region məlumatı olmayan mağazaları tap» → `LEFT JOIN ... WHERE right.key IS NULL`.
</details>

In [ ]:
q("""
SELECT s.store,
       COALESCE(st.region, 'Unknown') AS region,
       ROUND(SUM(s.revenue), 0)       AS revenue
FROM sales s
LEFT JOIN stores st ON s.store = st.store
GROUP BY s.store, st.region
ORDER BY revenue DESC
""")

### 11.3. Pəncərə funksiyaları (window functions)

<details><summary>Cavab</summary>

Dəyəri sətirlərin «pəncərəsi» üzrə hesablayır, onları **birləşdirmədən** (`GROUP BY`-dan fərqli olaraq).
* Sıralama: `ROW_NUMBER()` (1,2,3), `RANK()` (1,1,3), `DENSE_RANK()` (1,1,2).
* Aqreqatlar: `SUM() OVER (...)` — kumulyativ cəm, sürüşən orta.
* Sürüşmələr: `LAG()`, `LEAD()` — əvvəlki/növbəti sətrin dəyəri (keçən aya nisbətən artım).
* `PARTITION BY` — qruplar, `ORDER BY` — pəncərə daxilində ardıcıllıq.

Klassik tapşırıq: **hər qrupda top-N**.
</details>

In [ ]:
q("""
WITH store_cat AS (
    SELECT category, store, ROUND(SUM(revenue), 0) AS revenue
    FROM sales
    GROUP BY category, store
)
SELECT *
FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY category ORDER BY revenue DESC) AS rn
    FROM store_cat
) AS ranked
WHERE rn <= 2
""")

In [ ]:
q("""
WITH monthly AS (
    SELECT substr(date, 1, 7) AS month, ROUND(SUM(revenue), 0) AS revenue
    FROM sales
    GROUP BY month
)
SELECT month,
       revenue,
       SUM(revenue) OVER (ORDER BY month)    AS running_total,
       revenue - LAG(revenue) OVER (ORDER BY month) AS diff_vs_prev
FROM monthly
ORDER BY month
""").head(6)

### 11.4. Klassik tapşırıqlar: təkrarlar və «ikinci ən böyük dəyər»

In [ ]:
print(q("""
SELECT order_id, COUNT(*) AS cnt
FROM sales_raw
GROUP BY order_id
HAVING COUNT(*) > 1
LIMIT 5
"""))

print(q("""
SELECT MAX(price) AS second_max_price
FROM sales
WHERE price < (SELECT MAX(price) FROM sales)
"""))

### 11.5. Verilənlər bazaları üzrə nəzəri suallar

<details><summary>Cavablar</summary>

* **Primary key** — sətrin unikal identifikatoru; **foreign key** — başqa cədvəlin açarına istinad (əlaqələrin bütövlüyü).
* **İndeks** (adətən B-tree) axtarışı və JOIN-i sürətləndirir, amma əlavə etmə/yeniləmə əməliyyatlarını yavaşladır və yer tutur.
* **Normallaşdırma** — məlumat təkrarının aradan qaldırılması (1NF, 2NF, 3NF); analitik anbarlarda çox vaxt **denormallaşdırırlar** («ulduz» sxemi: faktlar cədvəli + ölçü-sorğu cədvəlləri).
* **Tranzaksiyalar və ACID:** Atomicity, Consistency, Isolation, Durability.
* **OLTP və OLAP:** əməliyyat tranzaksiyaları və analitik sorğular.
* **`UNION` və `UNION ALL`:** birincisi təkrarları silir (daha yavaş).
* **`COUNT(*)` və `COUNT(col)`:** ikincisi NULL-ları saymır.
* **SQL və NoSQL:** ciddi sxem və JOIN-lər, yoxsa çevik sənədlər/açar-dəyər (MongoDB, Redis).
* Python-dan: `sqlalchemy` + `pd.read_sql`; SQL inyeksiyalarına qarşı parametrləşdirilmiş sorğular.
</details>

---
# 12. Deep Learning, PyTorch / TensorFlow, LLM

### 12.1. Neyron şəbəkələrinin əsas anlayışları

<details><summary>Cavab</summary>

* **Neyron:** girişlərin çəkili cəmi + sürüşmə (bias) → aktivasiya funksiyası.
* **Aktivasiyalar:** ReLU (gizli qatlarda standart), Sigmoid (binar çıxış üçün ehtimal), Softmax (çoxsinifli çıxış), Tanh, GELU (transformerlərdə). Qeyri-xəttilik olmadan şəbəkə xətti modelə ekvivalentdir.
* **İtki funksiyaları:** MSE (reqressiya), Binary Cross-Entropy, Categorical Cross-Entropy.
* **Backpropagation:** zəncir qaydası ilə itki funksiyasının bütün çəkilərə görə qradiyentlərinin hesablanması; sonra optimallaşdırıcının addımı (SGD, Adam).
* **Epoch** — bütün məlumatlar üzrə bir keçid; **batch** — bir addım üçün məlumat hissəsi.
* **Overfitting-ə qarşı:** dropout, weight decay (L2), early stopping, data augmentation, batch normalization (həmçinin öyrətməni sabitləşdirir).
* **Sönən / partlayan qradiyentlər:** ReLU, düzgün inisializasiya, BatchNorm, residual əlaqələr, gradient clipping.
</details>

### 12.2. Hansı tapşırıqlar üçün hansı arxitekturalar?

<details><summary>Cavab</summary>

* **MLP (tam əlaqəli şəbəkə)** — cədvəl məlumatları (baxmayaraq ki, cədvəllərdə boosting adətən daha güclüdür).
* **CNN** — şəkillər (bükmələr lokal nümunələri tapır): rəfdə malların tanınması, qiymət etiketlərinin oxunması.
* **RNN / LSTM / GRU** — ardıcıllıqlar, zaman sıraları.
* **Transformer** (self-attention mexanizmi) — mətnlər, LLM, bu gün həm də şəkillər (ViT) və zaman sıraları.
* **Autoencoder** — sıxılma, anomaliya axtarışı.
* **Transfer learning:** əvvəlcədən öyrədilmiş modeli (ResNet, BERT) götürüb öz məlumatlarımızda əlavə öyrədirik — xeyli az məlumat lazım olur.
</details>

### 12.3. PyTorch və TensorFlow

<details><summary>Cavab</summary>

* **PyTorch:** dinamik qraf, «Python-a xas» üslub, tədqiqatlarda və Hugging Face ekosistemində üstünlük təşkil edir.
* **TensorFlow / Keras:** yüksək səviyyəli Keras API (`model.compile / fit`), TF Serving, mobil cihazlar üçün TF Lite.
* Hər ikisi GPU-nu və ONNX-ə ixracı dəstəkləyir. Junior vəzifədə frameworkün bütün detallarını yox, öyrətmə dövrünü başa düşmək vacibdir.
</details>

In [ ]:
Xm, ym = make_moons(n_samples=1000, noise=0.2, random_state=42)
Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.2, random_state=42)
mlp = MLPClassifier(hidden_layer_sizes=(16, 16), max_iter=2000, random_state=42).fit(Xm_tr, ym_tr)
print("sklearn MLP accuracy:", accuracy_score(ym_te, mlp.predict(Xm_te)))

In [ ]:
try:
    import torch
    import torch.nn as nn
except ImportError:
    torch = None
    print("PyTorch quraşdırılmayıb: pip install torch")

if torch is not None:
    torch.manual_seed(42)
    net = nn.Sequential(
        nn.Linear(2, 16), nn.ReLU(),
        nn.Linear(16, 16), nn.ReLU(),
        nn.Linear(16, 1),
    )
    loss_fn = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(net.parameters(), lr=0.01)

    X_t = torch.tensor(Xm_tr, dtype=torch.float32)
    y_t = torch.tensor(ym_tr, dtype=torch.float32).unsqueeze(1)

    for epoch in range(300):
        net.train()
        optimizer.zero_grad()
        loss = loss_fn(net(X_t), y_t)
        loss.backward()
        optimizer.step()
        if epoch % 100 == 0:
            print(f"epoch {epoch:3d}  loss={loss.item():.4f}")

    net.eval()
    with torch.no_grad():
        logits = net(torch.tensor(Xm_te, dtype=torch.float32))
        preds = (torch.sigmoid(logits) > 0.5).int().squeeze(1).numpy()
    print("PyTorch accuracy:", accuracy_score(ym_te, preds))

### 12.4. LLM və müasir AI haqqında suallar (AI Developer vəzifəsi üçün)

<details><summary>Cavablar</summary>

* **LLM** — böyük mətn korpusunda növbəti tokeni proqnozlaşdırmaq üçün əvvəlcədən öyrədilmiş, sonra təlimatlara əməl etmək üçün əlavə öyrədilmiş transformerdir (SFT, RLHF).
* **Token** — sözün bir hissəsi; kontekst pəncərəsi token sayı ilə məhdudlaşır; API-nin qiyməti tokenlərlə hesablanır.
* **Embeddings** — mətnin/obyektin vektor təsviri; mənaca yaxın mətnlər yaxın vektorlar verir (kosinus oxşarlığı). Tətbiq: semantik axtarış, tövsiyələr, rəylərin klasterləşməsi.
* **RAG (Retrieval-Augmented Generation):** sənədlər hissələrə (chunk) bölünür → embeddings → vektor bazası (FAISS, pgvector, Chroma) → sorğuya görə uyğun hissələr tapılır və LLM-in promptuna verilir. Halüsinasiyaları azaldır və şirkətin daxili məlumatları ilə işləməyə imkan verir.
* **Fine-tuning, RAG və prompt engineering:** promptlarla başlayırlar, sonra biliklər üçün RAG, fine-tuning (LoRA) — üslub/format və ya dar tapşırıq üçün.
* **Halüsinasiyalar** — inamlı, amma yanlış cavablar; mübarizə üsulları: RAG, mənbələrə istinad, yoxlamalar, aşağı temperature.
* **Agentlər / tool calling** — LLM tapşırığı həll etmək üçün funksiyaları (axtarış, SQL, API) çağırır.
* **Risklər:** şəxsi məlumatların xarici API-lərə sızması, prompt injection, xərc, cavab gecikməsi.
</details>

---
# 13. Modellərin inteqrasiyası: FastAPI, REST, MLOps

### 13.1. ML modelini mövcud tətbiqə necə inteqrasiya etmək olar?

<details><summary>Cavab</summary>

1. Bütün pipeline-ı **seriallaşdırmaq** (joblib / ONNX), kitabxana versiyalarını qeyd etmək.
2. Rejimi seçmək:
   * **Online (real vaxt):** REST API (FastAPI) — hər sorğuya proqnoz (kassada müştəri skorinqi).
   * **Batch:** gecə skripti / Airflow proqnozları hesablayıb bazaya yazır (bütün mallar üçün həftəlik tələb proqnozu).
   * **Streaming:** Kafka — hadisələrin real vaxtda emalı.
3. **Docker**-ə qablaşdırmaq, yerləşdirmək (Kubernetes, bulud), CI/CD.
4. **Giriş məlumatlarının validasiyası** (Pydantic), sorğu və cavabların loqlanması.
5. **Monitorinq:** servisin gecikməsi və xətaları, əlamətlərin paylanması (**data drift**), real etiketlər gəldikdən sonra modelin keyfiyyəti (**concept drift**), yenidən öyrətmə planı.
6. Modellərin və eksperimentlərin versiyalaşdırılması (MLflow), tam əvəzləmədən əvvəl yeni modelin A/B testi.
</details>

### 13.2. FastAPI servisi nümunəsi

Kodu `app.py` faylında saxla və işə sal: `uvicorn app:app --reload`. Swagger sənədləri `http://127.0.0.1:8000/docs` ünvanında olacaq.

```python
from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib
import pandas as pd

app = FastAPI(title="Churn prediction API")
model = joblib.load("churn_model.joblib")

class Customer(BaseModel):
    tenure_months: int = Field(ge=0)
    monthly_spend: float = Field(ge=0)
    purchases_last_90d: int = Field(ge=0)
    city: str
    has_loyalty_card: str

class Prediction(BaseModel):
    churn_probability: float

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict", response_model=Prediction)
def predict(customer: Customer):
    df = pd.DataFrame([customer.model_dump()])
    proba = float(model.predict_proba(df)[0, 1])
    return Prediction(churn_probability=round(proba, 4))
```

Sorğu: `curl -X POST http://127.0.0.1:8000/predict -H "Content-Type: application/json" -d '{"tenure_months": 3, "monthly_spend": 50, "purchases_last_90d": 1, "city": "Baku", "has_loyalty_card": "no"}'`

### 13.3. REST API haqqında suallar

<details><summary>Cavablar</summary>

* **REST** — arxitektura üslubu: URL ilə resurslar, standart HTTP metodları, sorğular arasında serverdə vəziyyətin saxlanmaması (stateless), adətən JSON.
* **Metodlar:** `GET` (oxuma), `POST` (yaratma / proqnoz çağırışı), `PUT` (tam əvəzləmə), `PATCH` (qismən dəyişiklik), `DELETE`. `GET`, `PUT`, `DELETE` — idempotentdir.
* **Cavab kodları:** 200 OK, 201 Created, 400 Bad Request, 401 Unauthorized, 403 Forbidden, 404 Not Found, 422 Unprocessable Entity (FastAPI-də validasiya xətası), 500 Internal Server Error, 503 Service Unavailable.
* **Niyə FastAPI:** sürətli (ASGI, async), Pydantic vasitəsilə validasiya, avtomatik OpenAPI/Swagger sənədləri.
* **`async def`** giriş-çıxış əməliyyatları üçün faydalıdır; ağır CPU inferensini adi `def`-də (FastAPI onu thread pool-da işə salacaq) və ya ayrıca worker-lərdə icra etmək daha yaxşıdır.
* **Təhlükəsizlik:** API açarları / OAuth2 / JWT, HTTPS, rate limiting, sirləri kodda saxlamamaq (mühit dəyişənləri, `.env` `.gitignore`-da).
</details>

---
# 14. Git / GitHub

<details><summary>Əsas əmrlər və suallar</summary>

| Əmr | Nə edir |
|---|---|
| `git clone <url>` | repozitoriyanı kopyalamaq |
| `git status`, `git diff` | nə dəyişib |
| `git add <file>` / `git commit -m "..."` | dəyişiklikləri hazırlamaq və qeyd etmək |
| `git switch -c feature/churn-model` | branch yaratmaq və ona keçmək |
| `git push -u origin feature/churn-model` | branch-ı GitHub-a göndərmək |
| `git fetch` | dəyişiklikləri yükləmək, birləşdir**mədən** |
| `git pull` | `fetch` + `merge` (və ya `rebase`) |
| `git merge` və `git rebase` | merge tarixçəni merge-commit ilə saxlayır; rebase commit-ləri başqa branch-ın üzərinə yenidən yazır (xətti tarixçə, ortaq branch-larla etmək olmaz) |
| `git stash` | commit olunmamış dəyişiklikləri müvəqqəti gizlətmək |
| `git log --oneline --graph` | tarixçə |
| `git revert <hash>` | commit-i yeni commit ilə ləğv etmək (təhlükəsiz) |
| `git reset` | branch göstəricisini köçürmək (dəyişiklikləri silə bilər — ehtiyatlı olun) |

* **Birləşmə konflikti (merge conflict):** Git hissələri `<<<<<<<`, `=======`, `>>>>>>>` ilə işarələyir; düzgün variantı seçirik, sonra `git add` və `git commit`.
* **Pull Request:** dəyişikliklər ayrıca branch-da → PR → code review → `main`-ə merge.
* **ML layihəsində nəyi commit etməmək lazımdır** (`.gitignore`): böyük məlumatlar, model çəkiləri, parol və açarlı `.env`, `__pycache__`, `.ipynb_checkpoints`. Məlumatlar və modellər üçün — DVC, Git LFS, MLflow.
* **Git-də notebook-lar:** commit-dən əvvəl output-ları təmizləmək (`nbstripout`), təkrar istifadə olunan kodu `.py` modullarına köçürmək.
* **Yaxşı commit mesajı:** qısa və konkret, əmr formasında: «Add lag features to demand model».
</details>

---
# 15. Keyslər və davranış sualları

### 15.1. «Pərakəndə şəbəkə mallara tələbi proqnozlaşdırmaq istəyir. Sənin planın?»

<details><summary>Nümunə cavab (CRISP-DM üzrə)</summary>

1. **Biznesi anlamaq:** proqnoz nəyə lazımdır (satınalmalar, out-of-stock və silinmələrin azaldılması), horizont (gün/həftə), səviyyə (mal × mağaza), xətanın qiyməti, uğur meyarı (WAPE cari metoddan X% aşağı).
2. **Məlumatlar:** çeklərdən satışlar (SQL), qiymətlər və aksiyalar, qalıqlar, təqvim və bayramlar, mağazaların açılışı.
3. **Hazırlıq:** boş dəyərlər, qalığın sıfır olduğu günlər (tələb satışa bərabər deyil), qaytarmalar, kənar dəyərlər.
4. **Baseline:** seasonal naive / sürüşən orta.
5. **Model:** lag-lar, sürüşən statistikalar, təqvim, qiymətlər üzərində LightGBM; zamana görə validasiya (backtesting).
6. **Qiymətləndirmə:** mal seqmentləri üzrə WAPE/MAE, baseline və cari əl ilə proqnozla müqayisə.
7. **Tətbiq:** gecə batch proqnozu → bazada cədvəl → Power BI hesabatı / satınalma sisteminə inteqrasiya.
8. Keyfiyyətin **monitorinqi** və müntəzəm yenidən öyrətmə.
</details>

### 15.2. «Model 99% accuracy göstərir. Reaksiyan?»

<details><summary>Cavab</summary>

Şübhə. Yoxlamaq lazımdır: siniflərin balansı (model həmişə çoxluğu proqnozlaşdırmır ki?), məlumat sızması (gələcəkdən əlamətlər, əlamətlərdə hədəf, train və test arasında təkrarlar), bölmənin düzgünlüyü (zamana görə, müştərilərə görə), precision/recall və xəta matrisinə baxmaq, dummy modellə müqayisə etmək.
</details>

### 15.3. «Testdə model yaxşıdır, production-da isə pis. Niyə?»

<details><summary>Cavab</summary>

* **Data drift:** giriş məlumatlarının paylanması dəyişib (yeni mağazalar, mövsüm, inflyasiya).
* **Concept drift:** asılılığın özü dəyişib (rəqiblərin yeni aksiyaları, müştəri davranışının dəyişməsi).
* **Training-serving skew:** production-da əlamətlər öyrətmədəkindən fərqli hesablanır.
* Öyrətmədə reallıqda olmayan sızma; reprezentativ olmayan test.
* Məlumat problemləri: boş dəyərlər, yeni kateqoriyalar, formatlarda xətalar.

Həll: monitorinq, öyrətmə və inferens üçün əlamətlərin vahid kodu (Pipeline / feature store), müntəzəm yenidən öyrətmə.
</details>

### 15.4. Digər tez-tez verilən suallar

<details><summary>Məsləhətlər</summary>

* **«ML layihən haqqında danış»** — STAR strukturu: tapşırıq → məlumatlar → nə etdin → metrika / nəticə (rəqəmlərlə) → nə öyrəndin. GitHub linkini hazır saxla və kodu göstərməyə hazır ol.
* **Kaggle:** hansı yarış, hansı həll, ən böyük artımı nə verdi (adətən model yox, əlamətlər və düzgün validasiya).
* **«Modeli menecerə necə izah edərsən?»** — jarqonsuz: nəyi proqnozlaşdırır, biznes terminlərində nə qədər dəqiqdir («gündə orta hesabla 8 ədəd mal qədər səhv edirik»), hansı faktorlar daha vacibdir, məhdudiyyətlər.
* **«Model, yoxsa qayda?»** — sadə qayda tapşırığı həll edirsə, qaydadan başlamaq; ML — qanunauyğunluqlar mürəkkəb olduqda və məlumat kifayət qədər olduqda.
* **«Necə öyrənirsən?»** — sənədlər (scikit-learn, PyTorch), məqalələr, kurslar, Kaggle, öz layihələrin. Texniki sənədləri ingilis dilində oxuduğunu qeyd et.
* **«Tapşırığı necə həll edəcəyini bilmirsənsə, nə edirsən?»** — hissələrə bölürəm, sənədlərdə/ədəbiyyatda axtarıram, sadə baseline qururam, komanda ilə məsləhətləşirəm, günlərlə tək qalmıram.
* **İşəgötürənə sənin sualların:** ML ilə hansı tapşırıqlar artıq həll olunur, hansı məlumatlar mövcuddur və harada saxlanılır, hansı stack (Python versiyası, bulud, verilənlər bazası), modellər production-a necə çatır, mentor kim olacaq, ilk 3 ay necə keçəcək.
</details>

---
# 16. Müsahibədən əvvəl yoxlama siyahısı

**Əminliklə izah edə bilməlisən:**
- [ ] train/val/test, kross-validasiya, data leakage
- [ ] overfitting / underfitting, L1/L2 regulyarizasiyası
- [ ] logistic regression, decision tree, random forest, gradient boosting, k-means
- [ ] bagging və boosting
- [ ] precision / recall / F1 / ROC-AUC, accuracy niyə aldadır
- [ ] MAE / RMSE / MAPE / R²
- [ ] boş dəyərlərin, kənar dəyərlərin, kateqorial əlamətlərin emalı, miqyaslama
- [ ] sklearn-də Pipeline + ColumnTransformer
- [ ] zaman sıralarının xüsusiyyətləri, baseline, lag-lar, `TimeSeriesSplit`
- [ ] Isolation Forest və anomaliya axtarışının digər metodları
- [ ] SQL: JOIN, GROUP BY / HAVING, pəncərə funksiyaları, icra ardıcıllığı
- [ ] neyron şəbəkəsinin öyrətmə dövrü (forward, loss, backward, step)
- [ ] modeli FastAPI ilə necə yerləşdirmək, data drift nədir
- [ ] Git: branch-lar, merge və rebase, pull request, konflikt

**İpucusuz yaza bilməlisən:**
- [ ] pandas-da `groupby().agg()` və `merge`
- [ ] kross-validasiyalı sklearn pipeline
- [ ] «hər qrupda top-N» SQL sorğusu
- [ ] sadə FastAPI endpoint

**Hazırla:**
- [ ] GitHub-da README ilə 1–2 öz layihən (tapşırıq, məlumatlar, metrikalar, necə işə salmaq)
- [ ] layihə haqqında 2 dəqiqəlik təqdimat
- [ ] «niyə bu şirkət və niyə ML» sualına cavab

**Faydalı resurslar:** scikit-learn sənədləri (User Guide), Kaggle Learn (Pandas, ML, Time Series), Andrew Ng-in «Machine Learning Specialization» kursu, «Hands-On Machine Learning» kitabı (Aurélien Géron), SQLBolt / LeetCode SQL 50, PyTorch və FastAPI-in rəsmi təlimatları.

**Müsahibədə uğurlar!**